In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import mean_squared_error

url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv"
df = pd.read_csv(url).fillna(0)

df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,0.0,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,0.0,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [5]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

y_train = df_train.fuel_efficiency_mpg.values
y_val = df_val.fuel_efficiency_mpg.values

del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']



In [21]:
dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(df_train.to_dict(orient='records'))
X_val = dv.transform(df_val.to_dict(orient='records'))

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

In [7]:
print(X_train.shape, X_val.shape)

(6000, 16) (2000, 16)


In [8]:
from sklearn.tree import DecisionTreeRegressor, export_text

dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)
print(export_text(dt, feature_names=list(dv.get_feature_names_out())))

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



In [10]:
dt = DecisionTreeRegressor(max_depth=2)
dt.fit(X_train, y_train)
print(export_text(dt, feature_names=list(dv.get_feature_names_out())))

|--- model_year <= 1997.50
|   |--- fuel_type=Gasoline <= 0.50
|   |   |--- value: [30.31]
|   |--- fuel_type=Gasoline >  0.50
|   |   |--- value: [27.83]
|--- model_year >  1997.50
|   |--- fuel_type=Gasoline <= 0.50
|   |   |--- value: [32.92]
|   |--- fuel_type=Gasoline >  0.50
|   |   |--- value: [30.37]



In [11]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_val)
print(rmse(y_val, y_pred))

1.837054054730018


In [13]:
for n in [10, 50, 100, 150]:
    rf = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)

    rf.fit(X_train, y_train)

    y_pred = rf.predict(X_val)
    print(n, round(rmse(y_val, y_pred), 3))

10 1.837
50 1.772
100 1.768
150 1.769


In [18]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

results = []

for depth in [10, 15, 20, 25]:
    scores = []

    for n in [10, 50, 100, 150]:
        model = RandomForestRegressor(
            n_estimators=n,
            max_depth=depth,
            random_state=1,
            n_jobs=-1
        )
        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        scores.append(rmse)

        print(f"depth={depth}, trees={n}, RMSE={rmse:.6f}")

    results.append({
        "max_depth": depth,
        "mean_rmse": np.mean(scores)
    })

summary = pd.DataFrame(results)
print("\nMean RMSE by depth:")
print(summary.to_string(index=False))

best = summary.loc[summary["mean_rmse"].idxmin()]
print(f"\nBest max_depth: {int(best['max_depth'])}")



depth=10, trees=10, RMSE=1.783693
depth=10, trees=50, RMSE=1.748567
depth=10, trees=100, RMSE=1.744839
depth=10, trees=150, RMSE=1.744520
depth=15, trees=10, RMSE=1.829892
depth=15, trees=50, RMSE=1.771141
depth=15, trees=100, RMSE=1.767874
depth=15, trees=150, RMSE=1.766904
depth=20, trees=10, RMSE=1.841031
depth=20, trees=50, RMSE=1.773109
depth=20, trees=100, RMSE=1.769152
depth=20, trees=150, RMSE=1.769676
depth=25, trees=10, RMSE=1.836775
depth=25, trees=50, RMSE=1.772108
depth=25, trees=100, RMSE=1.768403
depth=25, trees=150, RMSE=1.767983

Mean RMSE by depth:
 max_depth  mean_rmse
        10   1.755405
        15   1.783953
        20   1.788242
        25   1.786317

Best max_depth: 10


In [19]:
rf = RandomForestRegressor(n_estimators=10, max_depth=20,
                           random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

imp = pd.Series(rf.feature_importances_, index=dv.get_feature_names_out())
print(imp.sort_values(ascending=False).head(10))

model_year                      0.296001
vehicle_weight                  0.203879
fuel_type=Gasoline              0.160712
horsepower                      0.078017
engine_displacement             0.061005
acceleration                    0.055201
origin=USA                      0.042144
drivetrain=Front-wheel drive    0.020145
num_doors                       0.019415
fuel_type=Diesel                0.017038
dtype: float64


In [22]:
import xgboost as xgb

features = list(dv.get_feature_names_out())
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features)
watchlist = [(dtrain, 'train'), (dval, 'val')]

for eta in [0.3, 0.1]:
    xgb_params = {
        'eta': eta,
        'max_depth': 6,
        'min_child_weight': 1,
        'objective': 'reg:squarederror',
        'nthread': 8,
        'seed': 1,
        'verbosity': 1,
    }
    model = xgb.train(xgb_params, dtrain, num_boost_round=100,
                      evals=watchlist, verbose_eval=False)
    print(eta, round(rmse(y_val, model.predict(dval)), 3))

0.3 1.826
0.1 1.725
